In [ ]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
        print('{} memory growth: {}'.format(device, tf.config.experimental.get_memory_growth(device)))
else:
    print("Not enough GPU hardware devices available")

In [ ]:
import matplotlib.ticker as mtick 
from sklearn import tree
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter
import pickle 
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV

label_encoder = LabelEncoder()

output_dir = "../data/train/RFmodels/results"
os.makedirs(output_dir, exist_ok=True)
os.makedirs("../data/train/RFmodels/measure", exist_ok = True)
#Download csv file from Zenodo.
df = pd.read_csv("data/train/RFmodels/measure/RF_Culture_train.csv")

X = df.drop(["CellType", "No"], axis=1)
y = df["CellType"]
feature_names = X.columns

#k-fold
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

results = {}
confusion_matrices = [] 
sum_cm = np.zeros((len(set(y)),len(set(y))),dtype = int) 
all_importances = [] 
models = []

model_dir = os.path.join(output_dir, "models")
os.makedirs(model_dir, exist_ok=True)

for fold, (train_index, test_index) in enumerate(cv.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_classifier = RandomForestClassifier(n_estimators=300, max_features=0.8, random_state=42)
    rf_classifier.fit(X_train, y_train)
    models.append(rf_classifier) 

    y_pred = rf_classifier.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True)
    accuracy = accuracy_score(y_test, y_pred)
    max_depth = max(tree.get_depth() for tree in rf_classifier.estimators_)
    average_depth = np.mean([tree.get_depth() for tree in rf_classifier.estimators_])

    cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)
    confusion_matrices.append(cm.copy())
    sum_cm += cm 

    results[fold] = { 
        'fold': fold + 1,
        'classification_report': report,
        'accuracy': accuracy,
        'max_depth': max_depth,
        'average_depth': average_depth
    }
    
    with open(os.path.join(model_dir, f"RFmodel_fold{fold+1}.pkl"), 'wb') as f:
        pickle.dump(rf_classifier, f)

    with open(os.path.join(model_dir,f"RFmodel_fold{fold+1}.json"), 'w') as f: 
        json.dump(results, f, indent=4, default=str)

    for fold, result in results.items():
        df_report = pd.DataFrame(result['classification_report']).transpose()
        df_report['Max Depth'] = result['max_depth']
        df_report['Average Depth'] = result['average_depth']
        df_report.to_csv(os.path.join(output_dir, f"classification_report_fold{fold+1}.csv"), index=True)

    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - Fold {fold+1} (Counts)")
    plt.tight_layout()
    plt.rcParams["font.size"] = 16
    plt.savefig(os.path.join(output_dir, f"confusion_matrix_fold{fold+1}_counts.tif"), dpi=600, transparent=True)
    plt.close()

    cm_percentage = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100  
    cm_percentage = np.round(cm_percentage, decimals=2)  
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm_percentage, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
    
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - Fold {fold+1} (Percentage)")
    plt.tight_layout()
    plt.rcParams["font.size"] = 16
    plt.savefig(os.path.join(output_dir, f"confusion_matrix_fold{fold+1}_percentage.tif"), dpi=600, transparent=True)
    plt.close() 

    importances = rf_classifier.feature_importances_
    all_importances.append(importances) 
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10, 7))
    plt.title(f"Feature Importances - Fold {fold+1}")
    plt.barh(range(X.shape[1]), importances[indices], align="center", height=0.5) 
    plt.yticks(range(X.shape[1]), [feature_names[i] for i in indices]) 
    plt.ylim([-1, X.shape[1]]) 
    plt.gca().invert_yaxis()  
    
    plt.tight_layout()
    plt.rcParams["font.size"] = 12

    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    
    plt.savefig(os.path.join(output_dir, f"feature_importances_fold{fold+1}.tif"), dpi=600, transparent=True)
    plt.close()

    df_importances = pd.DataFrame({
        'feature': feature_names,  
        'importance': importances
    })

    df_importances = df_importances.sort_values('importance', ascending=False)

    df_importances.to_csv(os.path.join(output_dir, f"feature_importances_fold{fold+1}.csv"), index=False)  

with open(os.path.join(model_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

with open(os.path.join(model_dir, "feature_names.pkl"), "wb") as f:
    pickle.dump(feature_names, f)

average_importances = np.mean(all_importances, axis=0)

df_avg_importances = pd.DataFrame({
    'feature': feature_names,
    'average_importance': average_importances
})

df_avg_importances = df_avg_importances.sort_values('average_importance', ascending=False)
df_avg_importances.to_csv(os.path.join(output_dir, "average_feature_importances.csv"), index=False)

plt.figure(figsize=(10, 7))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Sum of Confusion Matrices (Counts)")
plt.tight_layout()
plt.rcParams["font.size"] = 16
plt.savefig(os.path.join(output_dir, "sum_confusion_matrix_counts.tif"), dpi=600, transparent=True)

sum_cm_percentage = sum_cm.astype('float') / sum_cm.sum(axis=1)[:, np.newaxis] * 100
sum_cm_percentage = np.round(sum_cm_percentage, decimals=2) 

plt.figure(figsize=(10, 7))
sns.heatmap(sum_cm_percentage, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Sum of Confusion Matrices (Percentage)")
plt.tight_layout()
plt.rcParams["font.size"] = 16
plt.savefig(os.path.join(output_dir, "sum_confusion_matrix_percentage.tif"), dpi=600, transparent=True)